# 🧠 Regularization
**Course:** Deep Learning  
**Lecture:** 4 — Optimizers and Regularization  
**Part:** b — Regularization

---

> **Where we are:** We know how to optimize our model. But if it optimizes too well on training data, it overfits. Now we learn to constrain it.

**What you'll learn:**
- L1 and L2 weight regularization
- Dropout and how it randomly drops activations
- Batch Normalization and its stabilizing effect
- Early Stopping to halt training at the right time

**Exam relevance:** ⭐⭐⭐ (Extremely relevant. You must be able to diagnose overfitting and choose the correct mitigation.)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 Theory: Regularization Techniques
When train loss $\ll$ validation loss, the model is overfitting.

- **L1 Regularization:** Adds $\lambda \sum |w|$ to the loss. Induces sparsity (many weights become zero).
- **L2 Regularization (Weight Decay):** Adds $\lambda \sum w^2$ to the loss. Shrinks weights evenly.
- **Dropout:** Randomly zeroes out neurons with probability $p$ during training, preventing co-adaptation.
- **Batch Normalization:** Normalizes layer inputs to mean 0, variance 1. Technically an optimization aid, but acts as a weak regularizer.
- **Early Stopping:** Halts training when validation metric stops improving.


## 🔀 Your Options
At this step, you can choose between several regularization strategies:

| Option | Pros | Cons | When to use |
|---|---|---|---|
| **None** | Baseline | Overfits | Prototyping |
| **L1/L2 Weight Decay** | Smooth bounds on weights | Tuning $\lambda$ required | Almost always (L2) |
| **Dropout** | Very effective anti-overfitting | Slower convergence | Dense layers, large models |
| **Batch Norm** | Faster training, allows larger LR | Small batch size issues | Deep CNNs/MLPs |
| **Early Stopping** | Prevents "over-training" | Might stop too early | Always (as a callback) |

In [ ]:
# Load IMDB data (reduced size to force overfitting)
vocab_size = 5000
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

# Pad sequences
maxlen = 100
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = tf.keras.preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)

# Subset to make it overfit easily
x_train, y_train = x_train[:2000], y_train[:2000]
x_test, y_test = x_test[:1000], y_test[:1000]


In [ ]:
from tensorflow.keras import regularizers

(x_train_raw, y_train), (x_test_raw, y_test) = tf.keras.datasets.imdb.load_data(num_words=10000)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train_raw, maxlen=256)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test_raw,  maxlen=256)

def make_base_layers(regularizer=None, dropout_rate=0.0, use_bn=False):
    """Build a Dense stack with optional L1/L2, Dropout, and BatchNorm."""
    lst = [
        tf.keras.layers.Embedding(10000, 32, input_length=256),
        tf.keras.layers.Flatten(),
    ]
    for units in [256, 128]:
        lst.append(tf.keras.layers.Dense(units, activation='relu',
                                         kernel_regularizer=regularizer))
        if use_bn:
            lst.append(tf.keras.layers.BatchNormalization())
        if dropout_rate > 0:
            lst.append(tf.keras.layers.Dropout(dropout_rate))
    lst.append(tf.keras.layers.Dense(1, activation='sigmoid'))
    return lst

model_baseline = tf.keras.Sequential(make_base_layers(),                                                              name='Baseline')
model_l1       = tf.keras.Sequential(make_base_layers(regularizer=regularizers.l1(0.001)),                           name='L1')
model_l2       = tf.keras.Sequential(make_base_layers(regularizer=regularizers.l2(0.001)),                           name='L2')
model_dropout  = tf.keras.Sequential(make_base_layers(dropout_rate=0.4),                                             name='Dropout')
model_bn       = tf.keras.Sequential(make_base_layers(use_bn=True),                                                  name='BatchNorm')
model_combined = tf.keras.Sequential(make_base_layers(regularizer=regularizers.l2(0.001), dropout_rate=0.3, use_bn=True), name='Combined')

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

models = {
    'Baseline': model_baseline, 'L1': model_l1, 'L2': model_l2,
    'Dropout': model_dropout,   'BatchNorm': model_bn, 'Combined': model_combined,
}

histories = {}
for name, model in models.items():
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cbs = [early_stop] if name == 'Combined' else []
    print(f'Training {name}...')
    h = model.fit(x_train, y_train, epochs=10, batch_size=512,
                  validation_split=0.2, callbacks=cbs, verbose=0)
    histories[name] = h
    va = max(h.history['val_accuracy']); ta = max(h.history['accuracy'])
    print(f'  train={ta:.3f}  val={va:.3f}  gap={ta-va:.3f}')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, h) in zip(axes.flat, histories.items()):
    ax.plot(h.history['accuracy'],     label='Train', lw=2)
    ax.plot(h.history['val_accuracy'], label='Val',   lw=2, linestyle='--')
    ax.set_title(name); ax.set_ylim(0.5, 1.0); ax.legend(); ax.set_xlabel('Epoch')
plt.suptitle('Effect of regularization on overfitting')
plt.tight_layout(); plt.show()

In [ ]:
### ✅ SOLUTION ###
def get_baseline():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, 32, input_length=maxlen),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

def get_l1():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, 32, input_length=maxlen),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l1(0.001)),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l1(0.001)),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

def get_l2():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, 32, input_length=maxlen),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

def get_dropout(rate=0.5):
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, 32, input_length=maxlen),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(rate),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(rate),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

def get_batchnorm():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, 32, input_length=maxlen),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Dense(64, use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

def get_combined():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, 32, input_length=maxlen),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, use_bias=False, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

models = {
    'Baseline': get_baseline(),
    'L1': get_l1(),
    'L2': get_l2(),
    'Dropout': get_dropout(0.5),
    'BatchNorm': get_batchnorm(),
    'Combined': get_combined()
}

histories = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    callbacks = []
    if name == 'Combined':
        callbacks.append(tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True))
        
    histories[name] = model.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test), callbacks=callbacks, verbose=0)


In [ ]:
### 🔬 Comparison Cell: Learning Curves & Weight Distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (name, hist) in enumerate(histories.items()):
    axes[i].plot(hist.history['loss'], label='Train Loss')
    axes[i].plot(hist.history['val_loss'], label='Val Loss')
    axes[i].set_title(f"{name}")
    axes[i].legend()

plt.tight_layout()
plt.show()

# Weight distribution comparison
w_base = models['Baseline'].layers[2].get_weights()[0].flatten()
w_l1 = models['L1'].layers[2].get_weights()[0].flatten()
w_l2 = models['L2'].layers[2].get_weights()[0].flatten()

plt.figure(figsize=(10, 5))
plt.hist(w_base, bins=50, alpha=0.5, label='Baseline', density=True)
plt.hist(w_l1, bins=50, alpha=0.5, label='L1 (Sparse)', density=True)
plt.hist(w_l2, bins=50, alpha=0.5, label='L2 (Shrunk)', density=True)
plt.title("Weight Distribution Comparison")
plt.legend()
plt.xlim(-0.2, 0.2)
plt.show()


## 🎓 Exam Corner

### Question 1 (Conceptual)
What is the difference between L1 and L2 regularization? Which produces sparser weights?

<details><summary>💡 Answer</summary>

L1 penalizes the absolute value of weights, pulling many exactly to zero, thus acting as feature selection (sparsity). L2 penalizes squared weights, which shrinks large weights heavily but rarely forces them entirely to zero. L1 produces sparser weights.

</details>

### Question 2 (Conceptual)
Where exactly is BatchNormalization applied in a network? (before or after activation?)

<details><summary>💡 Answer</summary>

The original paper recommends applying it *before* the non-linear activation (Dense -> BatchNorm -> ReLU). However, empirically, some modern architectures apply it *after* the activation. Before activation is the "textbook" answer.

</details>

### Question 3 (Practical)
Explain what Dropout does during training vs during inference.

<details><summary>💡 Answer</summary>

During training, it randomly zeroes out neurons with probability $p$. To keep the expected sum of activations the same, the active neurons are scaled by $1 / (1-p)$. During inference, Dropout is turned off (all neurons are active), and no scaling is applied (since it was handled at training time).

</details>

### Question 4 (Practical)
What does `restore_best_weights=True` do in EarlyStopping?

<details><summary>💡 Answer</summary>

When early stopping halts training because the validation metric hasn't improved for `patience` epochs, the model is at its final state. `restore_best_weights=True` reverts the model's weights to the state they were in at the epoch with the best validation score, rather than the final overfitted epoch.

</details>